# Clustering Validation — Stability, PCA Boundary Check & DBSCAN
**Purpose:** picks up directly where `01_EDA_and_Baseline_Clustering.ipynb` leaves off. That notebook already covers task-level EDA and the baseline K-means clustering (elbow, silhouette, k=3 interpretation) — this notebook does **not** repeat any of that. It loads the already-computed teammate cluster profile and tests how much that result can actually be trusted.

**Covers:**
1. Multi-seed stability analysis (Section 3.9)
2. PCA boundary check (Section 3.9)
3. DBSCAN with parameter sweep (Section 3.5)

**Requires:** `teammate_time_allocation_clusters.csv`, exported by `01_EDA_and_Baseline_Clustering.ipynb`'s final cell. Run that notebook first if this file doesn't exist yet.

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

INPUT_CSV = "teammate_time_allocation_clusters.csv"


## 1. Load the Baseline Clustering Output
Re-derives the standardized feature matrix from the already-computed category-share profile — this single transform step is trivial and not a repeat of notebook 01's actual EDA or elbow/silhouette scan.

In [8]:
labeled = pd.read_csv(INPUT_CSV, index_col=0)

baseline_cluster_col = "cluster"
feature_cols = [c for c in labeled.columns if c != baseline_cluster_col]

teammate_ids = labeled.index.tolist()
X = StandardScaler().fit_transform(labeled[feature_cols].values)
baseline_labels = pd.Series(labeled[baseline_cluster_col].values, index=teammate_ids)

print(f"Loaded {len(teammate_ids)} teammates, {X.shape[1]} features")
print("Baseline cluster sizes (from notebook 01):", baseline_labels.value_counts().to_dict())


Loaded 22 teammates, 10 features
Baseline cluster sizes (from notebook 01): {0: 19, 2: 2, 1: 1}


## 2. Multi-Seed Stability Analysis
Runs K-means across multiple seeds and k-values to test whether the baseline clustering's isolation of any individual reflects genuine structure or random initialization — per Section 3.9.

In [9]:
SEEDS = [1, 7, 42, 99, 2024]
K_VALUES = [2, 3, 4, 5]

flag_counts = {tid: 0 for tid in teammate_ids}
n_runs = 0

for k in K_VALUES:
    for seed in SEEDS:
        km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X)
        run_labels = pd.Series(km.labels_, index=teammate_ids)
        smallest = run_labels.value_counts().idxmin()
        flagged = run_labels[run_labels == smallest].index.tolist()
        for tid in flagged:
            flag_counts[tid] += 1
        n_runs += 1

stability_pct = (pd.Series(flag_counts) / n_runs * 100).round(1).sort_values(ascending=False)
print(f"Total runs: {n_runs} ({len(K_VALUES)} k-values x {len(SEEDS)} seeds)")
print("\nTeammates flagged as smallest-cluster member, by % of runs:")
display(stability_pct[stability_pct > 0])


Total runs: 20 (4 k-values x 5 seeds)

Teammates flagged as smallest-cluster member, by % of runs:


T-018    50.0
T-004    35.0
T-003    15.0
dtype: float64

### Alternative framing — checking a specific pair together
Rather than "whoever's in the smallest cluster," this checks a specific hypothesis: do two named individuals land in the *same* cluster as each other, across the same seed/k sweep. Useful if your finding is about a pair clustering together, not about singleton isolation.

In [10]:
PAIR_TO_CHECK = ("T-022", "T-023")  # edit as needed

together_count = 0
for k in K_VALUES:
    for seed in SEEDS:
        km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X)
        run_labels = pd.Series(km.labels_, index=teammate_ids)
        a, b = PAIR_TO_CHECK
        if a in run_labels.index and b in run_labels.index:
            if run_labels[a] == run_labels[b]:
                together_count += 1

print(f"{PAIR_TO_CHECK[0]} and {PAIR_TO_CHECK[1]} clustered together in "
      f"{together_count}/{n_runs} runs ({together_count/n_runs*100:.1f}%)")


T-022 and T-023 clustered together in 20/20 runs (100.0%)


## 3. PCA Boundary Check
Tests whether a flagged candidate sits clearly within its own cluster, or near a fragile boundary between clusters — per Section 3.9. Uses the baseline (k=3) clustering from notebook 01.

In [11]:
CANDIDATE = "T-023"  # change to test a different individual

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

centroids_pca = np.array([
    X_pca[baseline_labels.values == c].mean(axis=0) for c in sorted(baseline_labels.unique())
])

if CANDIDATE in teammate_ids:
    idx = teammate_ids.index(CANDIDATE)
    point = X_pca[idx]
    own_cluster = baseline_labels[CANDIDATE]
    distances = np.linalg.norm(centroids_pca - point, axis=1)

    print(f"{CANDIDATE} is in baseline cluster {own_cluster}")
    for c, d in zip(sorted(baseline_labels.unique()), distances):
        marker = " <- own cluster" if c == own_cluster else ""
        print(f"  Distance to cluster {c} centroid: {d:.2f}{marker}")
else:
    print(f"{CANDIDATE} not found.")


T-023 is in baseline cluster 2
  Distance to cluster 0 centroid: 3.22
  Distance to cluster 1 centroid: 7.18
  Distance to cluster 2 centroid: 0.95 <- own cluster


## 4. DBSCAN with Parameter Sweep
DBSCAN is known to be sensitive to its eps/minPts parameters (Karami & Johansson, 2014); rather than trusting one arbitrary pair, this sweeps a range and records each individual's noise-classification rate — per Section 3.5.

In [12]:
EPS_VALUES = [2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0]
MINPTS_VALUES = [2, 3, 4]

noise_counts = {tid: 0 for tid in teammate_ids}
n_settings = 0

for eps in EPS_VALUES:
    for min_pts in MINPTS_VALUES:
        db = DBSCAN(eps=eps, min_samples=min_pts).fit(X)
        db_labels = pd.Series(db.labels_, index=teammate_ids)
        noise_ids = db_labels[db_labels == -1].index.tolist()
        for tid in noise_ids:
            noise_counts[tid] += 1
        n_settings += 1

noise_pct = (pd.Series(noise_counts) / n_settings * 100).round(1)
print(f"Total parameter combinations tested: {n_settings} ({len(EPS_VALUES)} eps x {len(MINPTS_VALUES)} minPts)")
print("\nTeammates flagged as noise, by % of tested settings:")
display(noise_pct[noise_pct > 0].sort_values(ascending=False))


Total parameter combinations tested: 27 (9 eps x 3 minPts)

Teammates flagged as noise, by % of tested settings:


T-004    88.9
T-018    88.9
T-008    66.7
T-003    66.7
T-023    55.6
T-022    51.9
T-002    44.4
T-013    44.4
dtype: float64

## 5. Summary

Compare against Sections 3.5 and 3.9. Reference values from the original thesis analysis: K-means silhouette ≈ 0.364 (see notebook 01); T-018 flagged in ~89% of DBSCAN settings; T-023 flagged in ~50%.

**Note:** if the "smallest cluster" stability result (Section 2 above) doesn't match your thesis text, check the pair-specific result instead (the cell right after) — your original finding may have been framed around whether T-022 and T-023 cluster together, rather than singleton isolation alone.